# Projeto Prisma — Concepção e Arquitetura
## Decompondo o crescimento da ÍconeLab

**Lufi Data Consulting** · Segundo projeto de portfólio  
**Stack:** Python · PostgreSQL · Power BI  
**Período:** Mai/2026 → Jun/2026

---

### O que este notebook faz

Este notebook é o ponto de partida do Projeto Prisma. Execute as células **em ordem**, de cima para baixo — uma única vez.

| Célula | O que faz |
|---|---|
| 1 | Configuração: caminhos e parâmetros do projeto |
| 2 | Verificação do ambiente Python e bibliotecas |
| 3 | Criação da estrutura de pastas |
| 4 | Geração do `.gitignore` e `requirements.txt` |
| 5 | **Fase 1** — Geração das dimensões (`d_tempo`, `d_canal`, `d_loja`, `d_cliente`, `d_produto`) |
| 6 | **Fase 2** — Geração da fato de vendas (`f_vendas`, ~228 mil itens, 7 profit leaks injetados) |
| 7 | **Fase 3** — Geração das tabelas complementares (`f_marketing`, `f_estoque`, `f_devolucoes`) |
| 8 | Verificação final da estrutura e dos datasets |

---

### Estrutura que será criada

```
prisma/
├── 01 — Concepção e Arquitetura.ipynb   ← você está aqui
├── requirements.txt
├── .gitignore
├── data/
│   ├── dataset/          ← 9 CSVs gerados por este notebook
│   ├── sql/              ← arquivos .sql (schema, views, queries)
│   └── exports/          ← outputs visuais (.png, .xlsx)
├── notebooks/            ← notebooks de análise exploratória
├── dashboards/           ← arquivo .pbix do Power BI
├── docs/                 ← READMEs executivo e técnico
└── logs/                 ← relatórios de integridade
```

---

> **Pré-requisito:** Python 3.10 ou superior · `pip install -r requirements.txt`

---
## ⚙️ Célula 1 — Configuração

Defina abaixo o caminho onde o projeto está localizado.  
Use `'.'` se estiver executando diretamente da pasta raiz do projeto (recomendado).

> **Dica:** Use `/` mesmo no Windows. Exemplo: `C:/Users/seu_nome/projetos/prisma`

In [1]:
# ============================================================
#  CONFIGURAÇÃO — edite apenas esta célula
# ============================================================

# Caminho da raiz do projeto
# Deixe '.' para criar na pasta atual (onde este notebook está)
PASTA_PROJETO = '.'

# Nome do autor (para o README)
AUTOR_NOME = 'Luiz Filho'

# URL do LinkedIn (para o README)
AUTOR_LINKEDIN = 'https://linkedin.com/in/seu-perfil'

# ============================================================
#  Não mexa no código abaixo
# ============================================================
from pathlib import Path

BASE       = Path(PASTA_PROJETO)
PASTA_DATA = BASE / 'data' / 'dataset'   # CSVs gerados
PASTA_SQL  = BASE / 'data' / 'sql'       # arquivos .sql
PASTA_EXP  = BASE / 'data' / 'exports'  # outputs visuais
PASTA_LOGS = BASE / 'logs'

print('Projeto Prisma — ÍconeLab')
print(f'Raiz do projeto : {BASE.resolve()}')
print(f'Dataset (CSVs)  : {PASTA_DATA}')
print(f'Autor           : {AUTOR_NOME}')

Projeto Prisma — ÍconeLab
Raiz do projeto : C:\Users\LUFI\anaconda_projects\PRISMA
Dataset (CSVs)  : data\dataset
Autor           : Luiz Filho


---
## ✅ Célula 2 — Verificação do ambiente

Verifica Python e bibliotecas necessárias antes de qualquer geração de dado.

In [2]:
import sys
import importlib

# --- Python ---
versao = sys.version_info
print(f'Python detectado: {sys.version}')

if versao.major < 3 or (versao.major == 3 and versao.minor < 10):
    print('\n❌ ATENÇÃO: Este projeto requer Python 3.10 ou superior.')
    print('   Faça o download em: https://www.python.org/downloads/')
else:
    print(f'\n✅ Versão OK! Python {versao.major}.{versao.minor} é compatível.')

# --- Bibliotecas ---
print('\n📦 Verificando bibliotecas...')
libs = [
    ('pandas',  'pandas'),
    ('numpy',   'numpy'),
    ('pathlib', 'pathlib'),
]

todas_ok = True
for modulo, nome in libs:
    try:
        importlib.import_module(modulo)
        print(f'  ✅ {nome}')
    except ImportError:
        print(f'  ❌ {nome} — NÃO INSTALADO')
        print(f'     Execute: pip install {nome}')
        todas_ok = False

if todas_ok:
    print('\n✅ Ambiente pronto para geração do dataset.')
else:
    print('\n❌ Instale as bibliotecas em falta antes de continuar.')

Python detectado: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]

✅ Versão OK! Python 3.13 é compatível.

📦 Verificando bibliotecas...
  ✅ pandas
  ✅ numpy
  ✅ pathlib

✅ Ambiente pronto para geração do dataset.


---
## 📁 Célula 3 — Criação da estrutura de pastas

Cria todas as pastas do projeto. Se já existirem, **não serão sobrescritas** — seguro executar mais de uma vez.  
Arquivos `.gitkeep` são criados para que pastas vazias apareçam no Git.

In [3]:
from pathlib import Path

BASE = Path(PASTA_PROJETO)

# Estrutura de pastas do Projeto Prisma
# Derivada do padrão Lufi Data Consulting (cria_pastas_projetos.py)
# com ajustes para o contexto SQL + Power BI do Prisma
PASTAS = [
    'data/dataset',    # CSVs sintéticos gerados (equivalente a data/originais no FICTUS)
    'data/sql',        # schema.sql, views.sql, queries.sql — SQL core do projeto
    'data/exports',    # outputs visuais (.png, .xlsx exportados do Power BI ou Python)
    'notebooks',       # análises exploratórias e validações
    'dashboards',      # prisma_dashboard.pbix
    'docs',            # README_executivo.md, README_tecnico.md
    'logs',            # relatórios de integridade gerados pelos scripts
]

print('Criando estrutura de pastas...\n')

for pasta in PASTAS:
    caminho = BASE / pasta
    caminho.mkdir(parents=True, exist_ok=True)
    gitkeep = caminho / '.gitkeep'
    if not gitkeep.exists():
        gitkeep.touch()
        status = '✅ criada'
    else:
        status = '📂 já existia'
    print(f'  {status}: {pasta}/')

print('\n✅ Estrutura de pastas criada com sucesso!')

Criando estrutura de pastas...

  📂 já existia: data/dataset/
  📂 já existia: data/sql/
  📂 já existia: data/exports/
  📂 já existia: notebooks/
  📂 já existia: dashboards/
  📂 já existia: docs/
  📂 já existia: logs/

✅ Estrutura de pastas criada com sucesso!


---
## 🚫 Célula 4 — `.gitignore` e `requirements.txt`

O `.gitignore` protege dados gerados e arquivos sensíveis.  
Os CSVs do dataset são **gerados localmente** e não vão para o GitHub —  
repositório limpo + reprodutibilidade garantida via script com `SEED=42`.

In [4]:
from pathlib import Path
BASE = Path(PASTA_PROJETO)

# ── .gitignore ────────────────────────────────────────────────────────────────
gitignore = """\
# =============================================================
# Projeto Prisma | ÍconeLab — .gitignore
# =============================================================

# Dataset sintético — gerado localmente pelo notebook de concepção
# Não versionado: pesado + reprodutível via script com SEED=42
data/dataset/
*.csv

# Power BI — arquivo binário pesado, gerenciar externamente
*.pbix

# Outputs e exports gerados
data/exports/

# Jupyter
.ipynb_checkpoints/
*/.ipynb_checkpoints/

# Python
__pycache__/
*.py[cod]
*.egg-info/

# Sistema operacional
.DS_Store
Thumbs.db
desktop.ini

# Ambientes virtuais
venv/
.venv/
env/

# Chaves e variáveis de ambiente
.env
.env.*
secrets.py
"""

arquivo_gi = BASE / '.gitignore'
arquivo_gi.write_text(gitignore.strip(), encoding='utf-8')
print('✅ .gitignore criado')
print('   Os CSVs gerados em data/dataset/ NÃO serão versionados.')
print('   Para reproduzir o dataset: execute as Células 5, 6 e 7 deste notebook.\n')

# ── requirements.txt ──────────────────────────────────────────────────────────
requirements = """\
pandas>=1.5.0
numpy>=1.23.0
"""

arquivo_req = BASE / 'requirements.txt'
arquivo_req.write_text(requirements.strip(), encoding='utf-8')
print('✅ requirements.txt criado')
print()
print('Para instalar as dependências, execute no terminal:')
print('  pip install -r requirements.txt')

✅ .gitignore criado
   Os CSVs gerados em data/dataset/ NÃO serão versionados.
   Para reproduzir o dataset: execute as Células 5, 6 e 7 deste notebook.

✅ requirements.txt criado

Para instalar as dependências, execute no terminal:
  pip install -r requirements.txt


---
## 🏗️ Célula 5 — Fase 1: Geração das Dimensões

Gera as 5 tabelas de dimensão do Star Schema:

| Tabela | Linhas | Descrição |
|---|---|---|
| `d_tempo` | 1.096 | Calendário 2023-2025 com atributos analíticos |
| `d_canal` | 4 | Ecommerce, Marketplace, Loja Física, App |
| `d_loja` | 12 | CDs e lojas físicas com região |
| `d_cliente` | 40.000 | 84 cidades, 19 estados, tipo_cliente (B2C/B2B) |
| `d_produto` | 700 | nome_produto, custo, taxa devolução, giro esperado |

**Output:** 5 CSVs em `data/dataset/`  
**Reprodutibilidade:** `SEED=42` — resultados idênticos em qualquer máquina.

In [5]:
import pandas as pd
import numpy as np
import random
from pathlib import Path

# ── Configuração ──────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PASTA = Path(PASTA_PROJETO) / 'data' / 'dataset'
PASTA.mkdir(parents=True, exist_ok=True)

N_CLIENTES  = 40_000
N_PRODUTOS  = 700
DATA_INICIO = pd.Timestamp('2023-01-01')
DATA_FIM    = pd.Timestamp('2025-12-31')

def wpick(lista, probs, n):
    return np.random.choice(lista, size=n, p=probs)

print('Iniciando Fase 1 — Geração das Dimensões')
print(f'Destino: {PASTA.resolve()}\n')

# ── d_tempo ───────────────────────────────────────────────────────────────────
datas = pd.date_range(DATA_INICIO, DATA_FIM, freq='D')

MESES_PT = {1:'Janeiro',2:'Fevereiro',3:'Março',4:'Abril',5:'Maio',
            6:'Junho',7:'Julho',8:'Agosto',9:'Setembro',
            10:'Outubro',11:'Novembro',12:'Dezembro'}

d_tempo = pd.DataFrame({'data': datas})
d_tempo['ano']           = d_tempo['data'].dt.year
d_tempo['trimestre']     = d_tempo['data'].dt.quarter
d_tempo['mes']           = d_tempo['data'].dt.month
d_tempo['nome_mes']      = d_tempo['mes'].map(MESES_PT)
d_tempo['semana']        = d_tempo['data'].dt.isocalendar().week.astype(int)
d_tempo['dia_semana']    = d_tempo['data'].dt.dayofweek + 1
d_tempo['fim_de_semana'] = d_tempo['dia_semana'].isin([6,7]).astype(int)
d_tempo['ano_mes']       = d_tempo['data'].dt.to_period('M').astype(str)

# ── d_canal ───────────────────────────────────────────────────────────────────
d_canal = pd.DataFrame({
    'id_canal':           [1,           2,             3,             4],
    'canal':              ['Ecommerce', 'Marketplace', 'Loja Física', 'App'],
    'margem_base_pct':    [0.32,         0.19,          0.28,          0.30],
    'desconto_medio_pct': [0.08,         0.18,          0.06,          0.10],
})

# ── d_loja ────────────────────────────────────────────────────────────────────
lojas = [
    (1,  'CD',   'São Paulo',      'SP', 'Sudeste'),
    (2,  'CD',   'Extrema',        'MG', 'Sudeste'),
    (3,  'Loja', 'Rio de Janeiro', 'RJ', 'Sudeste'),
    (4,  'Loja', 'Belo Horizonte', 'MG', 'Sudeste'),
    (5,  'Loja', 'Curitiba',       'PR', 'Sul'),
    (6,  'Loja', 'Porto Alegre',   'RS', 'Sul'),
    (7,  'Loja', 'Salvador',       'BA', 'Nordeste'),
    (8,  'Loja', 'Recife',         'PE', 'Nordeste'),
    (9,  'Loja', 'Brasília',       'DF', 'Centro-Oeste'),
    (10, 'Loja', 'Goiânia',        'GO', 'Centro-Oeste'),
    (11, 'CD',   'Manaus',         'AM', 'Norte'),
    (12, 'Loja', 'Fortaleza',      'CE', 'Nordeste'),
]
d_loja = pd.DataFrame(lojas,
    columns=['id_loja','tipo_unidade','cidade','estado','regiao'])

# ── d_cliente ─────────────────────────────────────────────────────────────────
cidades_por_estado = {
    'SP': ['São Paulo','Campinas','Santos','Ribeirão Preto','Sorocaba','Guarulhos'],
    'RJ': ['Rio de Janeiro','Niterói','Duque de Caxias','Nova Iguaçu','Petrópolis'],
    'MG': ['Belo Horizonte','Uberlândia','Contagem','Juiz de Fora','Montes Claros'],
    'ES': ['Vitória','Vila Velha','Cariacica','Serra'],
    'PR': ['Curitiba','Londrina','Maringá','Ponta Grossa','Cascavel'],
    'SC': ['Florianópolis','Joinville','Blumenau','Chapecó','Itajaí'],
    'RS': ['Porto Alegre','Caxias do Sul','Pelotas','Canoas','Santa Maria'],
    'BA': ['Salvador','Feira de Santana','Vitória da Conquista','Camaçari','Itabuna'],
    'PE': ['Recife','Caruaru','Olinda','Petrolina','Paulista'],
    'CE': ['Fortaleza','Caucaia','Juazeiro do Norte','Maracanaú','Sobral'],
    'MA': ['São Luís','Imperatriz','Caxias','Timon'],
    'GO': ['Goiânia','Aparecida de Goiânia','Anápolis','Rio Verde'],
    'DF': ['Brasília','Taguatinga','Ceilândia','Samambaia'],
    'MT': ['Cuiabá','Várzea Grande','Rondonópolis','Sinop'],
    'MS': ['Campo Grande','Dourados','Três Lagoas','Corumbá'],
    'AM': ['Manaus','Parintins','Itacoatiara','Manacapuru'],
    'PA': ['Belém','Ananindeua','Santarém','Marabá'],
    'RO': ['Porto Velho','Ji-Paraná','Ariquemes'],
    'AC': ['Rio Branco','Cruzeiro do Sul','Sena Madureira'],
}
estados_regiao = {
    'SP':'Sudeste','RJ':'Sudeste','MG':'Sudeste','ES':'Sudeste',
    'PR':'Sul','SC':'Sul','RS':'Sul',
    'BA':'Nordeste','PE':'Nordeste','CE':'Nordeste','MA':'Nordeste',
    'GO':'Centro-Oeste','DF':'Centro-Oeste','MT':'Centro-Oeste','MS':'Centro-Oeste',
    'AM':'Norte','PA':'Norte','RO':'Norte','AC':'Norte',
}

estado_lista = list(cidades_por_estado.keys())
peso_estados = np.array([
    0.22,0.12,0.10,0.03,
    0.06,0.04,0.06,
    0.06,0.04,0.05,0.02,
    0.04,0.03,0.02,0.02,
    0.02,0.02,0.02,0.01,
])
peso_estados /= peso_estados.sum()

estados_clientes = np.random.choice(estado_lista, size=N_CLIENTES, p=peso_estados)
cidades_clientes = [random.choice(cidades_por_estado[e]) for e in estados_clientes]
regioes_clientes = [estados_regiao[e] for e in estados_clientes]

dias_total     = (DATA_FIM - DATA_INICIO).days
dias_cadastro  = np.random.choice(range(dias_total - 1), size=N_CLIENTES)
datas_cadastro = [DATA_INICIO + pd.Timedelta(days=int(d)) for d in dias_cadastro]

tipo_cliente = wpick(
    ['B2C Individual', 'B2C Premium', 'B2B Corporativo'],
    [0.78, 0.12, 0.10],
    N_CLIENTES
)

d_cliente = pd.DataFrame({
    'id_cliente':       range(1, N_CLIENTES + 1),
    'tipo_cliente':     tipo_cliente,
    'sexo':             wpick(['F','M','Outro'], [0.55,0.43,0.02], N_CLIENTES),
    'faixa_etaria':     wpick(['18-24','25-34','35-44','45-54','55+'],
                              [0.10,0.30,0.30,0.20,0.10], N_CLIENTES),
    'cidade':           cidades_clientes,
    'estado':           estados_clientes,
    'regiao':           regioes_clientes,
    'data_cadastro':    datas_cadastro,
    'segmento_cliente': wpick(
        ['Novo','Recorrente','VIP','Inativo'],
        [0.30, 0.42, 0.13, 0.15],
        N_CLIENTES
    ),
    'canal_origem': wpick(
        ['Orgânico','Pago','Indicação','Marketplace'],
        [0.30, 0.35, 0.20, 0.15],
        N_CLIENTES
    ),
})

# ── d_produto ─────────────────────────────────────────────────────────────────
categorias = {
    'Tech & Lifestyle':      (['Wearables','Audiovisual','Home Tech'],                        0.26, 0.12),
    'Wellness & Beauty':     (['Niche Beauty','Home Fragrance','Autocuidado'],                0.48, 0.04),
    'Home & Living':         (['Têxteis','Hospitalidade','Organização de Closet'],            0.34, 0.06),
    'Fashion & Accessories': (['Vestuário Premium','Artigos de Couro','Relojoaria & Joalheria'], 0.42, 0.08),
    'Gourmet & Experience':  (['Cutelaria & Cozinha','Barware & Mixologia','Empório Curado'], 0.31, 0.05),
}

marcas = [
    'Maison Véra','Aurum Studio','Norde','Calla','Sève',
    'Lumis','Brasa','Orin','Vael','Celso Mori'
]

nomes_por_sub = {
    'Wearables':              ['Smartwatch Orin Série S','Pulseira Vael Carbon','Fone Norde Arc Pro',
                               'Case Couro Vael AirPods','Rastreador Lumis Slim'],
    'Audiovisual':            ['Toca-discos Brasa Vintage','Caixa Hi-Fi Aurum Studio','Fone Celso Mori Over-Ear',
                               'Rádio Digital Norde Oak','Câmera Rangefinder Lumis'],
    'Home Tech':              ['Estação Carga Mármore Vael','Luminária Sève Arc','Difusor Inteligente Calla',
                               'Carregador Latão Aurum','Speaker Norde Slim'],
    'Niche Beauty':           ['Perfume Sève Noir 100ml','Sérum Calla Vitamin C','Kit Skincare Maison Véra',
                               'Suplemento Lumis Clean','Óleo Facial Orin Rose'],
    'Home Fragrance':         ['Vela Sève Âmbar 200g','Difusor Calla Bambu','Kit Velas Maison Véra',
                               'Home Spray Norde Cedar','Vela Brasa Vetiver'],
    'Autocuidado':            ['Roupão Algodão Pima Vael','Kit Spa Maison Véra','Esfoliante Calla Marine',
                               'Óleo Corporal Lumis Gold','Toalha Premium Orin'],
    'Têxteis':                ['Jogo Cama Egípcio Vael','Toalha Banho Orin 700g','Manta Cashmere Maison Véra',
                               'Almofada Linho Norde','Roupão Teka Lumis'],
    'Hospitalidade':          ['Louça Exclusiva Calla Set','Taça Cristal Aurum','Jogo Americano Sève',
                               'Porta-Guardanapo Brasa','Centro Mesa Vael Latão'],
    'Organização de Closet':  ['Cabide Veludo Vael 6un','Caixa Linho Maison Véra','Sapateira Norde Modular',
                               'Organizador Couro Brasa','Kit Gaveta Lumis'],
    'Vestuário Premium':      ['Blazer Lã Merino Celso Mori','Camisa Linho Norde','Trench Coat Aurum',
                               'Calça Alfaiataria Vael','Malha Seda Maison Véra'],
    'Artigos de Couro':       ['Bolsa Couro Maison Véra','Cinto Artesanal Brasa','Carteira Vael Slim',
                               'Mochila Celso Mori','Calçado Orin Handmade'],
    'Relojoaria & Joalheria': ['Relógio Aurum Clássico','Colar Prata Calla','Pulseira Orin Gold',
                               'Brinco Lumis Statement','Anel Maison Véra'],
    'Cutelaria & Cozinha':    ['Faca Damasco Norde Chef','Tábua Nogueira Brasa','Kit Facas Aurum 5pç',
                               'Pinça Inox Celso Mori','Moedor Cerâmica Vael'],
    'Barware & Mixologia':    ['Coqueteleira Ouro Aurum','Copo Cristal Sève Set6','Fôrma Gelo Lumis',
                               'Abridor Brasa Artesanal','Kit Bar Maison Véra'],
    'Empório Curado':         ['Azeite Gran Reserva Calla','Vinho Orgânico Norde','Chocolate Origem Brasa',
                               'Kit Charcuterie Lumis','Mel Trufa Maison Véra'],
}

prod_rows = []
for i in range(1, N_PRODUTOS + 1):
    cat = random.choice(list(categorias.keys()))
    subs, margem_b, devolucao_b = categorias[cat]
    sub   = random.choice(subs)
    marca = random.choice(marcas)
    nome  = random.choice(nomes_por_sub[sub])

    preco_raw      = np.random.lognormal(mean=6.5, sigma=0.9)
    preco          = round(np.clip(preco_raw, 80, 8000), 2)
    faixa          = ('Essencial' if preco < 500 else ('Signature' if preco < 2000 else 'Icon'))
    custo_pct      = 1 - margem_b + np.random.uniform(-0.05, 0.05)
    custo_unitario = round(preco * custo_pct, 2)
    taxa_devolucao = round(devolucao_b + np.random.uniform(0, 0.04), 4)
    giro_esperado  = round(np.random.uniform(0.5, 8.0), 2)

    prod_rows.append([
        i, cat, sub, marca, nome, preco, faixa,
        f'Fornecedor_{random.randint(1, 80)}',
        round(custo_unitario, 2), taxa_devolucao, giro_esperado
    ])

d_produto = pd.DataFrame(prod_rows, columns=[
    'id_produto','categoria','subcategoria','marca','nome_produto',
    'preco_lista','faixa_preco','fornecedor',
    'custo_unitario','taxa_devolucao_esperada','giro_esperado'
])

# ── Exportar ──────────────────────────────────────────────────────────────────
d_tempo.to_csv(   PASTA / 'd_tempo.csv',    index=False, encoding='utf-8-sig')
d_cliente.to_csv( PASTA / 'd_cliente.csv',  index=False, encoding='utf-8-sig')
d_produto.to_csv( PASTA / 'd_produto.csv',  index=False, encoding='utf-8-sig')
d_canal.to_csv(   PASTA / 'd_canal.csv',    index=False, encoding='utf-8-sig')
d_loja.to_csv(    PASTA / 'd_loja.csv',     index=False, encoding='utf-8-sig')

# ── Relatório Fase 1 ──────────────────────────────────────────────────────────
print('\n' + '═'*55)
print('  FASE 1 CONCLUÍDA — Dimensões | Projeto Prisma')
print('═'*55)
print(f'  d_tempo   → {len(d_tempo):>6,} linhas')
print(f'  d_cliente → {len(d_cliente):>6,} linhas')
print(f'  d_produto → {len(d_produto):>6,} linhas')
print(f'  d_canal   → {len(d_canal):>6,} linhas')
print(f'  d_loja    → {len(d_loja):>6,} linhas')
print(f'\n  ✓ Nenhum cliente com cadastro futuro : {(d_cliente["data_cadastro"] <= DATA_FIM).all()}')
print(f'  ✓ Produtos com custo definido        : {(d_produto["custo_unitario"] > 0).all()}')
print(f'  ✓ Produtos com nome definido         : {d_produto["nome_produto"].notna().all()}')
print(f'  ✓ Cidades únicas                     : {d_cliente["cidade"].nunique()}')
print(f'  ✓ Estados cobertos                   : {d_cliente["estado"].nunique()}')
print(f'\n  — Distribuição de preços (d_produto) —')
print(f'  Mínimo : R$ {d_produto["preco_lista"].min():>8,.2f}')
print(f'  Mediana: R$ {d_produto["preco_lista"].median():>8,.2f}')
print(f'  Média  : R$ {d_produto["preco_lista"].mean():>8,.2f}')
print(f'  Máximo : R$ {d_produto["preco_lista"].max():>8,.2f}')
print(f'\n  Faixas:')
print(d_produto['faixa_preco'].value_counts().to_string())
print(f'\n  — Mix de tipo_cliente —')
print(d_cliente['tipo_cliente'].value_counts().to_string())
print(f'\n✅  5 CSVs salvos em: {PASTA.resolve()}')

Iniciando Fase 1 — Geração das Dimensões
Destino: C:\Users\LUFI\anaconda_projects\PRISMA\data\dataset


═══════════════════════════════════════════════════════
  FASE 1 CONCLUÍDA — Dimensões | Projeto Prisma
═══════════════════════════════════════════════════════
  d_tempo   →  1,096 linhas
  d_cliente → 40,000 linhas
  d_produto →    700 linhas
  d_canal   →      4 linhas
  d_loja    →     12 linhas

  ✓ Nenhum cliente com cadastro futuro : True
  ✓ Produtos com custo definido        : True
  ✓ Produtos com nome definido         : True
  ✓ Cidades únicas                     : 84
  ✓ Estados cobertos                   : 19

  — Distribuição de preços (d_produto) —
  Mínimo : R$    80.00
  Mediana: R$   712.34
  Média  : R$   993.19
  Máximo : R$ 8,000.00

  Faixas:
faixa_preco
Signature    376
Essencial    254
Icon          70

  — Mix de tipo_cliente —
tipo_cliente
B2C Individual     31117
B2C Premium         4949
B2B Corporativo     3934

✅  5 CSVs salvos em: C:\Users\LUFI\anaconda_p

---
## 📊 Célula 6 — Fase 2: Geração da Fato de Vendas

Gera `f_vendas` com 120.000 pedidos e ~228.800 itens.  
Os **7 profit leaks** são injetados de forma controlada e rastreável:

| Leak | Mecanismo injetado |
|---|---|
| 1 — Desconto Marketplace | Multiplicador cresce 1.00 → 1.45 (2023→2025) |
| 2 — Frete Norte/Nordeste | Base Norte R$72 vs Sudeste R$18 · crescimento +38% |
| 3 — Devolução Tech | Taxa base 12% em Tech & Lifestyle · Marketplace amplifica ×1.35 |
| 6 — SLA Sazonal | Nov/Dez prazo base ×1.6 · piora YoY |
| 7 — Mix baixa margem | Tech & Lifestyle cresce: 18% → 22% → 26% do mix |

> Leaks 4 (CAC) e 5 (estoque) são injetados na Fase 3.

**Output:** `data/dataset/f_vendas.csv`

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

SEED = 42
rng  = np.random.default_rng(SEED)

PASTA = Path(PASTA_PROJETO) / 'data' / 'dataset'

d_cliente = pd.read_csv(PASTA / 'd_cliente.csv', parse_dates=['data_cadastro'])
d_produto = pd.read_csv(PASTA / 'd_produto.csv')
d_canal   = pd.read_csv(PASTA / 'd_canal.csv')
d_loja    = pd.read_csv(PASTA / 'd_loja.csv')

DATA_INICIO = pd.Timestamp('2023-01-01')
DATA_FIM    = pd.Timestamp('2025-12-31')
N_PEDIDOS   = 120_000

print('Iniciando Fase 2 — Geração de f_vendas')
print(f'Destino: {PASTA.resolve()}\n')

# ── Datas ponderadas (sazonalidade + crescimento YoY) ─────────────────────────
print('Gerando datas ponderadas...')
todos_dias = pd.date_range(DATA_INICIO, DATA_FIM, freq='D')
saz        = {1:0.75,2:0.72,3:0.85,4:0.88,5:0.90,6:0.87,
              7:0.92,8:0.95,9:0.98,10:1.05,11:1.45,12:1.35}
ano_mult   = {2023:1.00, 2024:1.12, 2025:1.27}
pesos      = np.array([saz[d.month] * ano_mult[d.year] for d in todos_dias], dtype=float)
pesos     /= pesos.sum()
idx_dias   = rng.choice(len(todos_dias), size=N_PEDIDOS, replace=True, p=pesos)
datas      = todos_dias[np.sort(idx_dias)]
anos       = datas.year.values
meses      = datas.month.values

# ── Canais — Leak 7: Marketplace cresce ──────────────────────────────────────
print('Gerando canais...')
canal_probs = {
    2023: [0.40, 0.20, 0.25, 0.15],
    2024: [0.36, 0.28, 0.22, 0.14],
    2025: [0.30, 0.38, 0.18, 0.14],
}
canais = np.zeros(N_PEDIDOS, dtype=int)
for a, p in canal_probs.items():
    mask = anos == a
    canais[mask] = rng.choice([1,2,3,4], size=mask.sum(), p=p)

# ── Clientes elegíveis por bucket temporal (coerência de cohort) ──────────────
print('Gerando clientes elegíveis...')
cad_dias   = ((d_cliente['data_cadastro'] - DATA_INICIO).dt.days.values).clip(0)
venda_dias = (datas - DATA_INICIO).days.values
d_cliente['bucket'] = (cad_dias // 90).clip(0, 11)
clientes_por_bucket = [
    d_cliente[d_cliente['bucket'] <= b].index.values
    for b in range(12)
]
pedido_bucket = (venda_dias // 90).clip(0, 11)
cli_ids    = np.zeros(N_PEDIDOS, dtype=int)
cli_regiao = np.empty(N_PEDIDOS, dtype=object)
cli_tipo   = np.empty(N_PEDIDOS, dtype=object)
for b in range(12):
    mask = pedido_bucket == b
    if mask.sum() == 0:
        continue
    pool   = clientes_por_bucket[b]
    chosen = rng.choice(pool, size=mask.sum(), replace=True)
    cli_ids[mask]    = d_cliente.loc[chosen, 'id_cliente'].values
    cli_regiao[mask] = d_cliente.loc[chosen, 'regiao'].values
    cli_tipo[mask]   = d_cliente.loc[chosen, 'tipo_cliente'].values

# ── Itens por pedido ──────────────────────────────────────────────────────────
print('Gerando itens por pedido...')
n_itens_por_pedido = rng.choice([1,2,3,4,5], size=N_PEDIDOS, p=[0.50,0.25,0.13,0.08,0.04])
pedido_ids = np.repeat(np.arange(1, N_PEDIDOS+1), n_itens_por_pedido)
anos_exp   = np.repeat(anos,       n_itens_por_pedido)
meses_exp  = np.repeat(meses,      n_itens_por_pedido)
canais_exp = np.repeat(canais,     n_itens_por_pedido)
datas_exp  = np.repeat(datas,      n_itens_por_pedido)
cli_exp    = np.repeat(cli_ids,    n_itens_por_pedido)
reg_exp    = np.repeat(cli_regiao, n_itens_por_pedido)
tipo_exp   = np.repeat(cli_tipo,   n_itens_por_pedido)
N_ITEMS    = len(pedido_ids)
print(f'Total de itens: {N_ITEMS:,}')

# ── Mix de categoria — Leak 7: Tech & Lifestyle cresce ───────────────────────
print('Gerando mix de categoria...')
cat_pesos_ano = {
    2023: {'Fashion & Accessories':0.30,'Tech & Lifestyle':0.18,
           'Wellness & Beauty':0.25,'Home & Living':0.17,'Gourmet & Experience':0.10},
    2024: {'Fashion & Accessories':0.27,'Tech & Lifestyle':0.22,
           'Wellness & Beauty':0.22,'Home & Living':0.17,'Gourmet & Experience':0.12},
    2025: {'Fashion & Accessories':0.24,'Tech & Lifestyle':0.26,
           'Wellness & Beauty':0.20,'Home & Living':0.16,'Gourmet & Experience':0.14},
}
cats_list      = list(cat_pesos_ano[2023].keys())
categorias_exp = np.empty(N_ITEMS, dtype=object)
for a, pw in cat_pesos_ano.items():
    mask = anos_exp == a
    categorias_exp[mask] = rng.choice(cats_list, size=mask.sum(), p=list(pw.values()))

# ── Produtos ──────────────────────────────────────────────────────────────────
print('Sorteando produtos por categoria...')
prod_ids = np.zeros(N_ITEMS, dtype=int)
precos   = np.zeros(N_ITEMS)
custos   = np.zeros(N_ITEMS)
for cat in cats_list:
    mask = categorias_exp == cat
    pool = d_produto[d_produto['categoria'] == cat]
    idx  = rng.choice(len(pool), size=mask.sum(), replace=True)
    prod_ids[mask] = pool['id_produto'].values[idx]
    precos[mask]   = pool['preco_lista'].values[idx]
    custos[mask]   = pool['custo_unitario'].values[idx]

# ── Descontos — Leak 1: Marketplace cresce 45% ───────────────────────────────
print('Calculando descontos...')
desc_base  = {1:0.06, 2:0.15, 3:0.04, 4:0.09}
desc_cresc = {2023:1.00, 2024:1.20, 2025:1.45}
desc_rates = (
    np.array([desc_base[c]  for c in canais_exp]) *
    np.array([desc_cresc[a] for a in anos_exp])   *
    rng.uniform(0.7, 1.4, N_ITEMS)
)
desc_rates    = np.clip(desc_rates, 0, 0.55)
qtd           = rng.choice([1,2,3], size=N_ITEMS, p=[0.75,0.20,0.05])
receita_bruta = precos * qtd
desc_valor    = desc_rates * precos * qtd
imposto       = precos * 0.12 * qtd
custo_total   = custos * qtd

# ── Fretes — Leak 2: Norte = 4× Sudeste, crescimento +38% ───────────────────
print('Calculando fretes...')
frete_base_regiao = {'Sudeste':18,'Sul':22,'Centro-Oeste':32,'Nordeste':52,'Norte':72}
frete_cresc       = {2023:1.00, 2024:1.18, 2025:1.38}
frete_canal       = {1:1.0, 2:1.0, 3:0.3, 4:1.0}
frete_val         = np.array([
    frete_base_regiao[r] * frete_cresc[a] * frete_canal[c] * rng.uniform(0.85, 1.25)
    for r, a, c in zip(reg_exp, anos_exp, canais_exp)
])

# ── Devoluções — Leak 3: Tech alta devolução ─────────────────────────────────
print('Calculando devoluções...')
dev_base  = {'Tech & Lifestyle':0.12,'Wellness & Beauty':0.03,'Home & Living':0.05,
             'Fashion & Accessories':0.08,'Gourmet & Experience':0.04}
dev_canal = {1:1.0, 2:1.35, 3:0.7, 4:1.1}
dev_cresc = {2023:1.0, 2024:1.15, 2025:1.30}
dev_rates = np.array([
    min(dev_base[cat] * dev_canal[c] * dev_cresc[a], 0.35)
    for cat, c, a in zip(categorias_exp, canais_exp, anos_exp)
])
devolvido = (rng.random(N_ITEMS) < dev_rates).astype(int)

# ── SLA — Leak 6: picos Nov/Dez degradando YoY ───────────────────────────────
print('Calculando SLA...')
sla_base  = {'Sudeste':3,'Sul':4,'Centro-Oeste':6,'Nordeste':9,'Norte':12}
sla_cresc = {2023:1.0, 2024:1.1, 2025:1.25}
sla_vals  = np.array([
    max(0, int(
        sla_base[r] * (1.6 if m in [11,12] else 1.0) *
        sla_cresc[a] * rng.uniform(0.8, 1.5)
    )) if c != 3 else 0
    for r, m, a, c in zip(reg_exp, meses_exp, anos_exp, canais_exp)
])

# ── Avaliações ────────────────────────────────────────────────────────────────
print('Calculando avaliações...')
av_normal = [0.04,0.06,0.15,0.35,0.40]
av_dev    = [0.30,0.35,0.20,0.10,0.05]
avaliacao = np.where(
    devolvido == 1,
    rng.choice([1,2,3,4,5], size=N_ITEMS, p=av_dev),
    rng.choice([1,2,3,4,5], size=N_ITEMS, p=av_normal)
)

# ── Lojas ─────────────────────────────────────────────────────────────────────
print('Sorteando lojas...')
lojas_dig = d_loja[d_loja['tipo_unidade'].isin(['CD','Loja'])]['id_loja'].values
lojas_fis = d_loja[d_loja['tipo_unidade'] == 'Loja']['id_loja'].values
loja_ids  = np.where(
    canais_exp == 3,
    rng.choice(lojas_fis, size=N_ITEMS, replace=True),
    rng.choice(lojas_dig, size=N_ITEMS, replace=True)
)

# ── Métricas calculadas ───────────────────────────────────────────────────────
print('Montando DataFrame final...')
receita_liq  = receita_bruta - desc_valor - frete_val - imposto
margem_bruta = receita_liq - custo_total
margem_pct   = np.where(receita_bruta > 0, margem_bruta / receita_bruta, 0)

f_vendas = pd.DataFrame({
    'id_pedido':          pedido_ids,
    'id_item':            np.arange(1, N_ITEMS+1),
    'data_venda':         datas_exp.date,
    'id_cliente':         cli_exp,
    'id_produto':         prod_ids,
    'id_canal':           canais_exp,
    'id_loja':            loja_ids,
    'categoria':          categorias_exp,
    'tipo_cliente':       tipo_exp,
    'regiao_cliente':     reg_exp,
    'qtd':                qtd,
    'preco_unitario':     precos.round(2),
    'desconto_valor':     desc_valor.round(2),
    'frete_valor':        frete_val.round(2),
    'custo_produto':      custo_total.round(2),
    'imposto_valor':      imposto.round(2),
    'receita_bruta':      receita_bruta.round(2),
    'receita_liquida':    receita_liq.round(2),
    'margem_bruta':       margem_bruta.round(2),
    'margem_pct':         margem_pct.round(4),
    'devolvido_flag':     devolvido,
    'prazo_entrega_dias': sla_vals,
    'avaliacao_cliente':  avaliacao,
})

f_vendas.to_csv(PASTA / 'f_vendas.csv', index=False, encoding='utf-8-sig')

# ── Relatório Fase 2 ──────────────────────────────────────────────────────────
print('\n' + '═'*55)
print('  FASE 2 CONCLUÍDA — f_vendas | Projeto Prisma')
print('═'*55)
print(f'  Pedidos únicos : {f_vendas["id_pedido"].nunique():>10,}')
print(f'  Itens totais   : {len(f_vendas):>10,}')
print(f'  Clientes únicos: {f_vendas["id_cliente"].nunique():>10,}')
por_ano      = f_vendas.copy()
por_ano['ano'] = pd.to_datetime(por_ano['data_venda']).dt.year
print('\n  — Receita, margem e pedidos por ano —')
res = por_ano.groupby('ano').agg(
    receita_bruta=('receita_bruta','sum'),
    margem_bruta=('margem_bruta','sum'),
    pedidos=('id_pedido','nunique'),
).round(0)
res['margem_pct_%']    = (res['margem_bruta'] / res['receita_bruta'] * 100).round(1)
res['receita_bruta_M'] = (res['receita_bruta'] / 1e6).round(2)
print(res[['receita_bruta_M','margem_pct_%','pedidos']].to_string())
print('\n  — Taxa de devolução por categoria —')
print(por_ano.groupby('categoria')['devolvido_flag'].mean().mul(100).round(1).to_string())
print('\n  — Frete médio por região —')
print(por_ano.groupby('regiao_cliente')['frete_valor'].mean().round(2).to_string())
print('\n  — Mix de canal (% pedidos) por ano —')
mix = por_ano.merge(d_canal[['id_canal','canal']], on='id_canal')
print(mix.groupby(['ano','canal'])['id_pedido'].nunique().unstack(fill_value=0).to_string())
print(f'\n✅  f_vendas.csv salvo — Projeto Prisma / ÍconeLab')

Iniciando Fase 2 — Geração de f_vendas
Destino: C:\Users\LUFI\anaconda_projects\PRISMA\data\dataset

Gerando datas ponderadas...
Gerando canais...
Gerando clientes elegíveis...
Gerando itens por pedido...
Total de itens: 228,808
Gerando mix de categoria...
Sorteando produtos por categoria...
Calculando descontos...
Calculando fretes...
Calculando devoluções...
Calculando SLA...
Calculando avaliações...
Sorteando lojas...
Montando DataFrame final...

═══════════════════════════════════════════════════════
  FASE 2 CONCLUÍDA — f_vendas | Projeto Prisma
═══════════════════════════════════════════════════════
  Pedidos únicos :    120,000
  Itens totais   :    228,808
  Clientes únicos:     32,592

  — Receita, margem e pedidos por ano —
      receita_bruta_M  margem_pct_%  pedidos
ano                                         
2023            86.46          16.1    35300
2024            97.47          11.9    39718
2025           109.59           6.9    44982

  — Taxa de devolução por cate

---
## 📦 Célula 7 — Fase 3: Tabelas Complementares

Gera as 3 tabelas que completam o modelo relacional:

| Tabela | Leak | Narrativa |
|---|---|---|
| `f_marketing` | 4 | CAC cresce 48% em 3 anos — eficiência de campanha cai |
| `f_estoque` | 5 + 6 | Gourmet com capital imobilizado · Tech/Fashion com ruptura no pico |
| `f_devolucoes` | 3 | Perda líquida oculta por categoria, motivo e destino do produto |

**Output:** `data/dataset/f_marketing.csv` · `f_estoque.csv` · `f_devolucoes.csv`

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

SEED = 42
rng  = np.random.default_rng(SEED)

PASTA = Path(PASTA_PROJETO) / 'data' / 'dataset'

f_vendas  = pd.read_csv(PASTA / 'f_vendas.csv',  parse_dates=['data_venda'])
d_produto = pd.read_csv(PASTA / 'd_produto.csv')
d_canal   = pd.read_csv(PASTA / 'd_canal.csv')

DATA_INICIO = pd.Timestamp('2023-01-01')
DATA_FIM    = pd.Timestamp('2025-12-31')
todos_dias  = pd.date_range(DATA_INICIO, DATA_FIM, freq='D')

print('Iniciando Fase 3 — Tabelas Complementares')
print(f'Destino: {PASTA.resolve()}\n')

# ══════════════════════════════════════════════════════════════════════════════
# f_marketing — Leak 4: CAC cresce ano a ano, eficiência cai
# ══════════════════════════════════════════════════════════════════════════════
print('Gerando f_marketing...')

campanhas = [
    ('Black Friday',             2, 11, 11, 200_000, 0.85),
    ('Natal & Presentes',        1, 12, 12, 160_000, 0.80),
    ('Dia das Mães',             1,  5,  5, 110_000, 0.78),
    ('Dia dos Namorados',        1,  6,  6,  80_000, 0.75),
    ('Dia dos Pais',             1,  8,  8,  70_000, 0.74),
    ('Semana do Design',         1,  3,  3,  55_000, 0.72),
    ('Ritual de Beleza',         4,  9,  9,  60_000, 0.73),
    ('Coleção Inverno',          1,  6,  7,  75_000, 0.70),
    ('Curadoria Gourmet',        2,  4,  4,  45_000, 0.65),
    ('Experiência Icon',         4,  1, 12,  35_000, 0.60),
    ('Frete Grátis Marketplace', 2,  1, 12,  28_000, 0.55),
    ('Liquidação Gourmet',       2,  7,  9,  38_000, 0.50),
]

mkt_rows    = []
campanha_id = 1

for ano in [2023, 2024, 2025]:
    cac_mult = {2023:1.00, 2024:1.22, 2025:1.48}[ano]
    inv_mult = {2023:1.00, 2024:1.18, 2025:1.35}[ano]

    for nome, canal_id, mes_ini, mes_fim, orc_base, efic_base in campanhas:
        dias_camp = [d for d in todos_dias
                     if d.year == ano and mes_ini <= d.month <= mes_fim]
        if not dias_camp:
            continue
        investimento_total = orc_base * inv_mult * rng.uniform(0.90, 1.10)
        inv_por_dia        = investimento_total / len(dias_camp)
        for dia in dias_camp:
            ruido        = rng.uniform(0.80, 1.20)
            investimento = round(inv_por_dia * ruido, 2)
            eficiencia   = max(0.30, efic_base / cac_mult * rng.uniform(0.85, 1.15))
            leads        = int(investimento * eficiencia * rng.uniform(0.08, 0.14))
            novos_cli    = int(leads * rng.uniform(0.12, 0.22))
            cac          = round(investimento / max(novos_cli, 1), 2)
            mkt_rows.append({
                'id_campanha':    campanha_id,
                'campanha':       nome,
                'data':           dia.date(),
                'ano':            dia.year,
                'mes':            dia.month,
                'id_canal':       canal_id,
                'investimento':   investimento,
                'leads':          leads,
                'novos_clientes': novos_cli,
                'cac':            cac,
                'eficiencia_pct': round(eficiencia * 100, 1),
            })
        campanha_id += 1

f_marketing = pd.DataFrame(mkt_rows)
f_marketing.to_csv(PASTA / 'f_marketing.csv', index=False, encoding='utf-8-sig')
print(f'  f_marketing: {len(f_marketing):,} linhas | {f_marketing["id_campanha"].nunique()} campanhas/ciclos')
print(f'  CAC médio por ano:')
print(f_marketing.groupby('ano')['cac'].mean().round(2).to_string())

# ══════════════════════════════════════════════════════════════════════════════
# f_estoque — Leak 5: Gourmet capital imobilizado | Leak 6: ruptura sazonal
# ══════════════════════════════════════════════════════════════════════════════
print('\nGerando f_estoque...')

datas_snapshot  = pd.date_range(DATA_INICIO, DATA_FIM, freq='W-MON')
estoque_inicial = {}
for _, row in d_produto.iterrows():
    pid  = int(row['id_produto'])
    giro = row['giro_esperado']
    base = int(500 / max(giro, 0.5) * rng.uniform(0.8, 1.2))
    estoque_inicial[pid] = base

f_vendas['semana_ini'] = (
    f_vendas['data_venda'].dt.to_period('W').apply(lambda p: p.start_time)
)
saidas_semana = (
    f_vendas.groupby(['id_produto','semana_ini'])['qtd']
    .sum().reset_index()
    .rename(columns={'semana_ini':'data','qtd':'saidas_reais'})
)
saidas_semana['data'] = pd.to_datetime(saidas_semana['data'])

est_rows      = []
estoque_atual = estoque_inicial.copy()

for data in datas_snapshot:
    ano          = data.year
    mes          = data.month
    pico_sazonal = mes in [11, 12]
    for _, prod in d_produto.iterrows():
        pid  = int(prod['id_produto'])
        cat  = prod['categoria']
        giro = prod['giro_esperado']
        mask         = ((saidas_semana['id_produto'] == pid) & (saidas_semana['data'] == data))
        saidas_reais = int(saidas_semana.loc[mask,'saidas_reais'].sum()) if mask.any() else 0
        est_ini      = estoque_atual[pid]
        threshold    = int(50 / max(giro, 0.5))
        if cat == 'Gourmet & Experience':
            reposicao = int(rng.uniform(80, 150)) if est_ini < threshold * 3 else 0
        elif pico_sazonal and cat in ['Tech & Lifestyle', 'Fashion & Accessories']:
            reposicao = int(rng.uniform(20, 60))
        else:
            reposicao = int(rng.uniform(30, 100)) if est_ini < threshold else 0
        est_fin            = max(0, est_ini + reposicao - saidas_reais)
        cobertura_dias     = (round((est_fin / max(saidas_reais / 7, 0.1)), 1)
        ruptura            = 1 if (est_fin == 0 and saidas_reais > 0) or (saidas_reais > 0 and cobertura_dias < 7) else 0
        custo_carregamento = round(est_fin * prod['custo_unitario'] * 0.003, 2)
                              if saidas_reais > 0 else 999.0)
        est_rows.append({
            'data':               data.date(),
            'ano':                ano,
            'mes':                mes,
            'id_produto':         pid,
            'categoria':          cat,
            'estoque_inicial':    est_ini,
            'entradas':           reposicao,
            'saidas':             saidas_reais,
            'estoque_final':      est_fin,
            'ruptura_flag':       ruptura,
            'custo_carregamento': custo_carregamento,
            'cobertura_dias':     min(cobertura_dias, 999.0),
        })
        estoque_atual[pid] = est_fin

f_estoque = pd.DataFrame(est_rows)
f_estoque.to_csv(PASTA / 'f_estoque.csv', index=False, encoding='utf-8-sig')
print(f'  f_estoque: {len(f_estoque):,} linhas')
print(f'  Taxa de ruptura por categoria (%):')
print(f_estoque.groupby('categoria')['ruptura_flag'].mean().mul(100).round(1).to_string())
print(f'  Custo de carregamento total por categoria:')
print(f_estoque.groupby('categoria')['custo_carregamento'].sum().round(0).sort_values(ascending=False).to_string())

# ══════════════════════════════════════════════════════════════════════════════
# f_devolucoes — custo oculto real de cada devolução
# ══════════════════════════════════════════════════════════════════════════════
print('\nGerando f_devolucoes...')

itens_devolvidos = f_vendas[f_vendas['devolvido_flag'] == 1].copy()

motivos_por_cat = {
    'Tech & Lifestyle':      (['Defeito técnico','Não funcionou como esperado',
                               'Produto diferente da descrição','Arrependimento de compra'],
                              [0.38, 0.32, 0.15, 0.15]),
    'Wellness & Beauty':     (['Reação alérgica','Fragrância diferente do esperado',
                               'Produto diferente da foto','Arrependimento de compra'],
                              [0.35, 0.28, 0.22, 0.15]),
    'Home & Living':         (['Produto com defeito','Cor diferente da foto',
                               'Tamanho não adequado','Qualidade abaixo do esperado'],
                              [0.28, 0.27, 0.25, 0.20]),
    'Fashion & Accessories': (['Tamanho incorreto','Cor diferente da foto',
                               'Qualidade abaixo do esperado','Arrependimento de compra'],
                              [0.40, 0.25, 0.20, 0.15]),
    'Gourmet & Experience':  (['Produto com defeito','Produto diferente da descrição',
                               'Prazo de validade próximo','Arrependimento de compra'],
                              [0.35, 0.30, 0.20, 0.15]),
}
custo_retorno_regiao = {
    'Sudeste':28,'Sul':34,'Centro-Oeste':45,'Nordeste':62,'Norte':80
}

dev_rows = []
for _, row in itens_devolvidos.iterrows():
    cat    = row['categoria']
    regiao = row['regiao_cliente']
    motivos_cat, probs_cat = motivos_por_cat[cat]
    motivo       = rng.choice(motivos_cat, p=probs_cat)
    custo_log    = round(custo_retorno_regiao[regiao] * rng.uniform(0.85, 1.25), 2)
    reembolso    = round(row['receita_bruta'] * rng.uniform(0.90, 1.00), 2)
    destino      = rng.choice(
        ['Revenda com desconto','Descarte','Devolução ao fornecedor'],
        p=[0.45, 0.25, 0.30]
    )
    valor_recuperado = round(
        reembolso * {'Revenda com desconto':0.40,
                     'Descarte':0.00,
                     'Devolução ao fornecedor':0.60}[destino], 2
    )
    perda_liquida = round(reembolso + custo_log - valor_recuperado, 2)
    dias_ate_dev  = int(rng.choice(
        [1,2,3,5,7,10,14,21,30],
        p=[0.05,0.08,0.12,0.15,0.20,0.15,0.12,0.08,0.05]
    ) * (0.7 if cat == 'Tech & Lifestyle' else 1.0))
    dev_rows.append({
        'id_item':                 int(row['id_item']),
        'id_pedido':               int(row['id_pedido']),
        'data_venda':              row['data_venda'].date(),
        'id_produto':              int(row['id_produto']),
        'id_canal':                int(row['id_canal']),
        'categoria':               cat,
        'regiao_cliente':          regiao,
        'motivo_devolucao':        motivo,
        'dias_ate_devolucao':      dias_ate_dev,
        'receita_perdida':         round(row['receita_bruta'], 2),
        'custo_logistico_retorno': custo_log,
        'reembolso_valor':         reembolso,
        'destino_produto':         destino,
        'valor_recuperado':        valor_recuperado,
        'perda_liquida':           perda_liquida,
    })

f_devolucoes = pd.DataFrame(dev_rows)
f_devolucoes.to_csv(PASTA / 'f_devolucoes.csv', index=False, encoding='utf-8-sig')
print(f'  f_devolucoes: {len(f_devolucoes):,} linhas')
perda = f_devolucoes.groupby('categoria')['perda_liquida'].agg(['sum','mean']).round(2)
perda.columns = ['perda_total','perda_media']
print(f'  Perda líquida por categoria:\n{perda.to_string()}')
print(f'\n  Motivos mais frequentes:')
print(f_devolucoes['motivo_devolucao'].value_counts().head(5).to_string())
print(f'\n✅  Fase 3 concluída — Tabelas complementares geradas.')

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2404965918.py, line 128)

---
## 🔍 Célula 8 — Verificação Final

Confirma que toda a estrutura foi criada corretamente e que os 9 CSVs estão presentes com os volumes esperados.

In [ ]:
from pathlib import Path
import pandas as pd

BASE  = Path(PASTA_PROJETO)
PASTA = BASE / 'data' / 'dataset'

print('=' * 60)
print('  Projeto Prisma | Verificação Final — Milestone 1')
print('=' * 60)

# ── Pastas ────────────────────────────────────────────────────────────────────
PASTAS_CHECAR = [
    'data/dataset/',
    'data/sql/',
    'data/exports/',
    'notebooks/',
    'dashboards/',
    'docs/',
    'logs/',
]
print('\n📁 Estrutura de pastas:')
pastas_ok = True
for p in PASTAS_CHECAR:
    existe = (BASE / p).exists()
    print(f'  {"✅" if existe else "❌"} {p}')
    if not existe:
        pastas_ok = False

# ── Arquivos de configuração ──────────────────────────────────────────────────
print('\n📄 Arquivos de configuração:')
arquivos_ok = True
for a in ['.gitignore', 'requirements.txt']:
    existe = (BASE / a).is_file()
    print(f'  {"✅" if existe else "❌"} {a}')
    if not existe:
        arquivos_ok = False

# ── CSVs gerados ──────────────────────────────────────────────────────────────
print('\n📊 Dataset — 9 CSVs em data/dataset/:')
csvs = [
    ('d_tempo',       1_096),
    ('d_cliente',    40_000),
    ('d_produto',       700),
    ('d_canal',           4),
    ('d_loja',           12),
    ('f_vendas',    228_000),   # ~228k — tolerância ±5%
    ('f_marketing',  15_000),
    ('f_estoque',   109_000),
    ('f_devolucoes', 24_000),
]
csvs_ok = True
for nome, esperado in csvs:
    caminho = PASTA / f'{nome}.csv'
    if caminho.is_file():
        n      = len(pd.read_csv(caminho))
        dentro = abs(n - esperado) / max(esperado, 1) < 0.15  # tolerância 15%
        status = '✅' if dentro else '⚠️ '
        print(f'  {status} {nome:<22} → {n:>9,} linhas')
    else:
        print(f'  ❌ {nome}.csv — NÃO ENCONTRADO')
        csvs_ok = False

# ── Resultado final ───────────────────────────────────────────────────────────
tudo_ok = pastas_ok and arquivos_ok and csvs_ok
print('\n' + '=' * 60)
if tudo_ok:
    print('  ✅ MILESTONE 1 CONCLUÍDO — Concepção e Arquitetura')
    print()
    print('  Próximos passos (M2 — SQL: Fundação):')
    print('  1. Criar o banco prisma no pgAdmin')
    print('  2. Conectar no DBeaver')
    print('  3. Executar data/sql/00_schema.sql')
    print('  4. Importar os 9 CSVs via \\COPY ou Import do DBeaver')
    print('  5. Executar data/sql/01_views_analiticas.sql')
    print('  6. Executar data/sql/02_queries_negocio.sql')
    print('  7. Executar data/sql/03_query_roi_leaks.sql  ← upgrade M2')
else:
    print('  ⚠️  Alguns itens precisam de atenção (veja ❌ acima).')
    print('     Execute as células anteriores antes de continuar.')
print('=' * 60)

---

### Estrutura final após execução deste notebook

```
prisma/
├── 01 — Concepção e Arquitetura.ipynb   ← você está aqui
├── requirements.txt                      ← gerado pela Célula 4
├── .gitignore                            ← gerado pela Célula 4
├── data/
│   ├── dataset/                          ← 9 CSVs gerados pelas Células 5-7
│   │   ├── d_tempo.csv         (1.096 linhas)
│   │   ├── d_cliente.csv      (40.000 linhas)
│   │   ├── d_produto.csv         (700 linhas)
│   │   ├── d_canal.csv             (4 linhas)
│   │   ├── d_loja.csv             (12 linhas)
│   │   ├── f_vendas.csv       (~228.800 linhas)
│   │   ├── f_marketing.csv     (~15.000 linhas)
│   │   ├── f_estoque.csv      (~109.000 linhas)
│   │   └── f_devolucoes.csv    (~24.000 linhas)
│   ├── sql/                              ← 00_schema · 01_views · 02_queries · 03_roi
│   └── exports/                          ← outputs visuais
├── notebooks/                            ← análises exploratórias
├── dashboards/                           ← prisma_dashboard.pbix
├── docs/                                 ← README_executivo · README_tecnico
└── logs/                                 ← relatórios de integridade
```

---

### Roadmap de Milestones

| Milestone | Foco | Upgrade |
|---|---|---|
| **M1** — Concepção e Arquitetura | Dataset sintético · 9 CSVs | — |
| **M2** — SQL: Fundação | Schema · ingestão · views | — |
| **M3** — SQL: Análise | 7 queries de diagnóstico | **+ #30 Query ROI dos leaks** |
| **M4** — Power BI: Modelo + Pág. 1-2 | Modelagem · DAX · Executive Summary | **+ #31 Deploy Supabase** |
| **M5** — Power BI: Pág. 3-5 + Entrega | Clientes · Operação · Plano de Ação | **+ #32 "Recomendação ao board"** |
| **M6** — Reputação & Voz do Mercado | NLP · scraping · página Power BI | Módulo independente |

---

### Reprodutibilidade

Os CSVs **não são versionados** no GitHub (`.gitignore`).  
Para reproduzir o dataset do zero em qualquer máquina:

```bash
pip install -r requirements.txt
# Execute este notebook de cima para baixo — Células 1 a 7
# SEED = 42 garante resultados idênticos
```

---

*Projeto Prisma — Lufi Data Consulting · Mai/2026*